In [0]:
from pyspark.sql import functions as F

# Configurações do arquivo e da tabela
path_csv = "/Workspace/Repos/seu_usuario/seu_repo/datasource/registro_viagens.csv" # Ajuste para o seu caminho no Databricks
table_name = "bronze_registro_viagens"

# 1. Leitura do CSV bruto do repositório
df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_csv)

# Padroniza o formato da coluna DATA
df_csv = df_csv.withColumn("DATA_DATE", F.to_date(F.col("DATA"), "dd/MM/yyyy"))

# 2. Verifica se a tabela Bronze já existe para aplicar a lógica incremental
if spark.catalog.tableExists(table_name):
    # Descobre a maior data já gravada na tabela Bronze
    max_date_row = spark.sql(f"SELECT MAX(TO_DATE(DATA, 'dd/MM/yyyy')) as max_data FROM {table_name}").collect()[0]
    max_data = max_date_row["max_data"]
    
    if max_data is not None:
        print(f"Última data encontrada na Bronze: {max_data}. Filtrando apenas registros posteriores...")
        # Filtra o CSV mantendo somente datas estritamente maiores que a última armazenada
        df_final = df_csv.filter(F.col("DATA_DATE") > max_data).drop("DATA_DATE")
    else:
        df_final = df_csv.drop("DATA_DATE")
else:
    print("Tabela Bronze não encontrada. Realizando carga inicial...")
    df_final = df_csv.drop("DATA_DATE")

# 3. Gravação inteligente (Append se houver novos dados, Overwrite se for a primeira criação)
if spark.catalog.tableExists(table_name) and max_data is not None:
    if df_final.count() > 0:
        df_final.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"Sucesso: {df_final.count()} novos registros adicionados (Append).")
    else:
        print("Nenhum dado novo encontrado. A tabela já está atualizada.")
else:
    df_final.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print("Tabela Bronze criada com sucesso (Carga Inicial).")